# 📸 Module 4 (Advanced): Periodic Snapshot Fact Table

## Overview

In this notebook we build a **periodic snapshot** fact table — one of the three classic fact table patterns defined by Ralph Kimball.

**What you'll learn:**
- Understand the periodic snapshot pattern and when to use it
- Build `gold.fact_daily_station_activity` aggregated at station-day grain
- Calculate net bike flow to support rebalancing analysis
- Query the snapshot to find supply/demand imbalances

---

**Prerequisites:**
- Completed Module 3 (Gold Layer)
- `silver_trips`, `gold.dim_date`, `gold.dim_start_station`, and `gold.dim_end_station` tables exist

## 🎓 Concept: Periodic Snapshot vs Transaction Fact

| Pattern | Grain | Rows grow how? | Best for |
|---------|-------|----------------|----------|
| **Transaction fact** | One row per event | Append-only | Detailed drill-down |
| **Periodic snapshot** | One row per entity per period | New slice each period | Trends over time, inventory, flow |
| Accumulating snapshot | One row per process instance | Updated in place | Pipeline / lifecycle tracking |

A periodic snapshot captures **where things stood** at the end of each period.  
For Oslo Bysykkel that means: *how many bikes left / arrived at each station on each day?*

This is extremely useful for the operations team who need to **rebalance** the bike fleet — move bikes from over-supplied stations to under-supplied ones.


## Step 1: Verify Source Tables

In [ ]:
%%sql
-- Verify silver_trips is ready
SELECT
    COUNT(*)                            AS total_trips,
    MIN(DATE(started_at))               AS min_date,
    MAX(DATE(started_at))               AS max_date,
    COUNT(DISTINCT start_station_id)    AS unique_stations
FROM silver_trips
WHERE is_valid = TRUE

## Step 2: Create `fact_daily_station_activity` (Periodic Snapshot)

In [ ]:
%%sql
-- ============================================================
-- FACT TABLE: fact_daily_station_activity  (Periodic Snapshot)
-- Grain : one row per station per calendar day
-- ============================================================
CREATE OR REPLACE TABLE gold.fact_daily_station_activity
USING DELTA
AS
WITH
-- Role-specific station dimensions from the Gold layer, conformed for this station-day snapshot
station_attributes AS (
    SELECT station_id, station_name, station_description, latitude, longitude, city_quadrant
    FROM gold.dim_start_station
    UNION
    SELECT station_id, station_name, station_description, latitude, longitude, city_quadrant
    FROM gold.dim_end_station
),
station_dimension AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY station_id) AS station_key,
        station_id,
        FIRST(station_name) AS station_name,
        FIRST(station_description) AS station_description,
        ROUND(AVG(latitude), 6) AS latitude,
        ROUND(AVG(longitude), 6) AS longitude,
        FIRST(city_quadrant) AS city_quadrant
    FROM station_attributes
    GROUP BY station_id
),
-- Trips that DEPARTED from each station on each day
departures AS (
    SELECT
        DATE(started_at)                        AS snap_date,
        start_station_id                        AS station_id,
        COUNT(*)                                AS trips_started,
        SUM(duration_seconds)                   AS total_duration_seconds_started,
        AVG(duration_seconds)                   AS avg_duration_seconds_started
    FROM silver_trips
    WHERE is_valid = TRUE
    GROUP BY DATE(started_at), start_station_id
),
-- Trips that ARRIVED at each station on each day
arrivals AS (
    SELECT
        DATE(ended_at)                          AS snap_date,
        end_station_id                          AS station_id,
        COUNT(*)                                AS trips_ended
    FROM silver_trips
    WHERE is_valid = TRUE
    GROUP BY DATE(ended_at), end_station_id
),
-- Union of all station-days (a station may appear only as origin or only as destination)
all_station_days AS (
    SELECT snap_date, station_id FROM departures
    UNION
    SELECT snap_date, station_id FROM arrivals
)
SELECT
    ROW_NUMBER() OVER (ORDER BY a.snap_date, a.station_id)              AS snapshot_key,

    -- Foreign keys and station attributes
    CAST(DATE_FORMAT(a.snap_date, 'yyyyMMdd') AS INT)                  AS date_key,
    ds.station_key,
    ds.station_id,
    ds.station_name,
    ds.station_description,
    ds.latitude,
    ds.longitude,
    ds.city_quadrant,

    -- Snapshot measures
    COALESCE(d.trips_started,  0)                                      AS trips_started,
    COALESCE(ar.trips_ended,   0)                                      AS trips_ended,

    -- Net flow: positive = more arrivals than departures (bikes accumulating)
    COALESCE(ar.trips_ended, 0) - COALESCE(d.trips_started, 0)         AS net_bike_flow,

    COALESCE(d.total_duration_seconds_started, 0)                      AS total_duration_seconds,
    ROUND(COALESCE(d.avg_duration_seconds_started, 0), 1)              AS avg_duration_seconds,

    -- Snapshot timestamp (end of day)
    TIMESTAMP(CONCAT(DATE_FORMAT(a.snap_date, 'yyyy-MM-dd'), ' 23:59:59')) AS snapshot_timestamp

FROM all_station_days a
JOIN station_dimension ds
    ON  a.station_id = ds.station_id
LEFT JOIN departures d
    ON  a.snap_date = d.snap_date AND a.station_id = d.station_id
LEFT JOIN arrivals ar
    ON  a.snap_date = ar.snap_date AND a.station_id = ar.station_id
ORDER BY a.snap_date, ds.station_key

## Step 3: Verify the Snapshot Table

In [ ]:
%%sql
SELECT
    COUNT(*)                        AS total_rows,
    COUNT(DISTINCT date_key)        AS snapshot_days,
    COUNT(DISTINCT station_id)      AS stations,
    SUM(trips_started)              AS total_departures,
    SUM(trips_ended)                AS total_arrivals
FROM gold.fact_daily_station_activity

## Step 4: Analytical Queries

In [ ]:
%%sql
-- Top 10 stations by total departures
SELECT
    station_name,
    SUM(trips_started)    AS total_departures,
    SUM(trips_ended)      AS total_arrivals,
    SUM(net_bike_flow)    AS cumulative_net_flow
FROM gold.fact_daily_station_activity
GROUP BY station_name
ORDER BY total_departures DESC
LIMIT 10

In [ ]:
%%sql
-- Most imbalanced stations (bikes always leaving, never returning)
SELECT
    station_name,
    SUM(net_bike_flow)    AS cumulative_net_flow,
    SUM(trips_started)    AS total_departures,
    SUM(trips_ended)      AS total_arrivals
FROM gold.fact_daily_station_activity
GROUP BY station_name
ORDER BY cumulative_net_flow ASC  -- most negative = bikes draining away
LIMIT 10

In [ ]:
%%sql
-- Daily total trips trend
SELECT
    dd.full_date,
    dd.day_name,
    SUM(f.trips_started)    AS total_departures,
    SUM(f.trips_ended)      AS total_arrivals
FROM gold.fact_daily_station_activity f
JOIN gold.dim_date dd USING (date_key)
GROUP BY dd.full_date, dd.day_name
ORDER BY dd.full_date

## 📌 Key Takeaways

- The **periodic snapshot** gives you a consistent, queryable history of state at a fixed interval
- Each period adds a **new set of rows** — rows are never updated
- `net_bike_flow` surfaces operational insights impossible to see in raw transaction data
- This pattern pairs well with **dim_date** for trend analysis over time
